# Carbon Tracing — Debug P1

Objectif : identifier pourquoi `compute_nodal_carbon_intensity` sort des CI France ~8 gCO₂/kWh au lieu de ~55 (cible RTE 2022).

## Stratégie

1. **Charger** le réseau depuis cache + RTE eco2mix 2022
2. **Dispatch** sur 1 semaine échantillon (168 h, rapide)
3. **Oracle CI directe** — sans tracing, juste Σ(EF × P_gen) / Σ(P_gen) — ça DOIT donner ~55 gCO₂/kWh sinon le problème est déjà au dispatch
4. **Tracing** — appel `compute_nodal_carbon_intensity`
5. **Test de conservation du carbone** : C_produit (Σ EF × P_gen) == C_consommé (Σ CI × P_load) ?
6. **Localiser la fuite** : par carrier, par zone, par nature de load (charge vs pompage vs pertes)
7. **Hypothèses de fix** et tests A/B

## Cible

Validation PASS = bilan carbone conservé à ±2 % ET CI France load-weighted ∈ [50, 60] gCO₂/kWh.

---
## 0. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import pypsa
import matplotlib.pyplot as plt

from carbon_model.plants.emission_factors import EMISSION_FACTORS, normalize_technology
from carbon_model.tracing.carbon_tracing import (
    compute_nodal_carbon_intensity,
    trace_carbon_snapshot,
    build_participation_matrix,
)
from carbon_model.config import get_path

RAW = get_path('raw_data')
PROCESSED = get_path('processed_data')
print(f'ROOT : {ROOT}')
print(f'RAW  : {RAW}')

---
## 1. Charger réseau (cache) + RTE eco2mix

In [ ]:
# --- Réseau depuis cache ---
net = pypsa.Network(str(PROCESSED / 'network_france.nc'))
print(f'Réseau chargé : {len(net.buses)} buses, {len(net.lines)} lignes, {len(net.generators)} generators')

# Buses FR
fr_buses = net.buses[net.buses.get('country', '') == 'FR'].index
print(f'  buses FR : {len(fr_buses)}')

# Répartition carriers
print('\nGénérateurs par carrier (top 10) :')
for c, n in net.generators['carrier'].value_counts().head(10).items():
    cap = net.generators[net.generators['carrier'] == c]['p_nom'].sum()
    print(f'  {c:20s} : {n:5d} gens, capacité totale = {cap/1000:.1f} GW')

In [ ]:
# --- RTE eco2mix 2022 ---
from carbon_model.downloaders.download_rte import RTEDownloader

rte_raw = RTEDownloader().load_eco2mix_csv(RAW / 'rte/eco2mix_2022..xls')
rte_h = rte_raw.select_dtypes(include='number').resample('h').mean()

print(f'RTE chargée : {len(rte_h)} h')
print(f'Période     : {rte_h.index[0]} → {rte_h.index[-1]}')
print(f'Colonnes CO2: {[c for c in rte_h.columns if "co2" in c.lower() or "rate" in c.lower()]}')

ci_col = 'co2_rate_gco2_kwh'
ci_rte_series = rte_h[ci_col].dropna()
print(f'\nCI RTE 2022 : mean={ci_rte_series.mean():.1f}, median={ci_rte_series.median():.1f}, max={ci_rte_series.max():.1f} gCO2/kWh')

---
## 2. Préparer dispatch sur 168 h échantillon

On travaille sur une semaine (rapide). Choix : 2022-01-10 → 2022-01-16 (semaine hiver avec gaz actif).

In [ ]:
# Échantillon
SAMPLE_START = pd.Timestamp('2022-01-10')
SAMPLE_END   = pd.Timestamp('2022-01-16 23:00')
snapshots = pd.date_range(SAMPLE_START, SAMPLE_END, freq='h')
print(f'Snapshots : {len(snapshots)} h sur {SAMPLE_START.date()} → {SAMPLE_END.date()}')

# Aligner snapshots réseau
net.set_snapshots(snapshots)

# Production nationale RTE (colonnes eco2mix)
carrier_col_map = {
    'nuclear_mw': 'nuclear', 'wind_mw': 'wind', 'solar_mw': 'solar',
    'hydro_mw': 'hydro', 'bioenergy_mw': 'biomass',
    'fossil_gas_mw': 'gas', 'fossil_oil_mw': 'oil', 'fossil_coal_mw': 'coal',
    'pumped_storage_mw': 'storage',
}
national_prod = pd.DataFrame(index=snapshots)
for eco_col, carrier in carrier_col_map.items():
    if eco_col in rte_h.columns:
        national_prod[carrier] = rte_h[eco_col].reindex(snapshots).clip(lower=0).fillna(0)
    else:
        print(f'  ⚠ colonne manquante : {eco_col}')

print(f'\nnational_prod : {national_prod.shape}')
print('Moyennes par carrier (GW) :')
for c in national_prod.columns:
    print(f'  {c:10s} : {national_prod[c].mean()/1000:7.2f} GW')

In [ ]:
# Dispatch capacity-weighted vers les bus
from carbon_model.simulation.dispatch import (
    disaggregate_national_to_nodes,
    apply_dispatch_to_network,
    disaggregate_load_to_nodes,
)

dispatch_df = disaggregate_national_to_nodes(net, national_prod)
apply_dispatch_to_network(net, dispatch_df)

# Conso
if 'consumption_mw' in rte_h.columns:
    conso = rte_h['consumption_mw'].reindex(snapshots).fillna(0)
    disaggregate_load_to_nodes(net, conso)

prod_total = net.generators_t.p_set.sum(axis=1)
load_total = net.loads_t.p_set.sum(axis=1) if len(net.loads) > 0 else pd.Series(0, index=snapshots)
print(f'Production dispatch      : {prod_total.mean()/1000:.1f} GW')
print(f'Consommation loads       : {load_total.mean()/1000:.1f} GW')
print(f'Résidu (prod - conso)    : {(prod_total.mean() - load_total.mean())/1000:+.2f} GW')

---
## 3. ✅ Audit dispatch : le fossile est-il bien présent ?

**Hypothèse A** : si les générateurs gaz/charbon ont `p_set = 0`, le tracing ne peut pas propager de fossile. Test ici.

In [ ]:
print('── Production dispatché par carrier (moyenne sample) ──')
prod_by_carrier = {}
for carrier in sorted(net.generators['carrier'].unique()):
    gen_ids = net.generators[net.generators['carrier'] == carrier].index
    p_mean = net.generators_t.p_set[gen_ids].sum(axis=1).mean()
    prod_by_carrier[carrier] = p_mean
    print(f'  {carrier:25s} : {p_mean/1000:6.2f} GW')

# Contrôle : fossile présent ?
fossile_total = sum(v for k, v in prod_by_carrier.items() if any(f in k for f in ['gas', 'coal', 'oil', 'lignite']))
print(f'\n── FOSSILE TOTAL : {fossile_total/1000:.2f} GW ──')
if fossile_total < 500:
    print('  ❌ FAIL : fossile < 0.5 GW, suspecté bug dispatch')
else:
    print(f'  ✅ OK : fossile présent ({fossile_total/1000:.1f} GW dispatché)')

---
## 4. 🎯 Oracle : CI production directe (SANS tracing)

Avant tout tracing, on calcule la CI nationale "brute" — simple moyenne pondérée par capacité dispatché.

**Si cette oracle ne donne pas ~55, le bug est au niveau du dispatch / EF, pas du tracing.**

In [ ]:
ef_vec = net.generators['carrier'].map(
    lambda c: EMISSION_FACTORS.get(c, EMISSION_FACTORS.get(normalize_technology(c), 300.0))
)
emissions_h = (net.generators_t.p_set * ef_vec).sum(axis=1)  # gCO2/h (au facteur MW près : MW × g/kWh = g/h  car 1 MW sur 1h = 1000 kWh)
# Unités : MW × gCO2/kWh = gCO2/h  (si on considère 1 MW·h/h = 10^3 kWh/h)... attention
# MW × g/kWh = (10^3 kWh/h) × g/kWh = 10^3 g/h = kg/h → on divise par 1000 pour g/h propres? 
# Mais ici on veut ratio sans unité, donc numérateur/dénominateur homogènes :
prod_h = net.generators_t.p_set.sum(axis=1)  # MW
ci_prod_direct = emissions_h / prod_h.replace(0, np.nan)  # g/kWh

print('── ORACLE — CI production directe (sans tracing) ──')
print(f'  mean          : {ci_prod_direct.mean():.2f} gCO2/kWh')
print(f'  median        : {ci_prod_direct.median():.2f}')
print(f'  min / max     : {ci_prod_direct.min():.1f} / {ci_prod_direct.max():.1f}')
print(f'\n── RTE référence (même fenêtre) ──')
ci_ref = ci_rte_series.reindex(snapshots)
print(f'  mean          : {ci_ref.mean():.2f} gCO2/kWh')
print(f'  ratio direct/RTE : {ci_prod_direct.mean() / ci_ref.mean():.2f}')

if 45 < ci_prod_direct.mean() < 75:
    print('\n  ✅ Oracle cohérente avec RTE. Dispatch OK — bug est DANS le tracing.')
else:
    print(f'\n  ❌ Oracle à {ci_prod_direct.mean():.1f} gCO2/kWh. Bug AVANT le tracing (dispatch, EF, ou mapping carrier).')

---
## 5. Exécuter le tracing

In [ ]:
import time

# Besoin d'un DC power flow — mais on peut trace sans flux (cas simple : aucun transit)
# D'abord essayer avec flux statiques = 0 (tracing pur "injection = consommation locale")
t0 = time.time()
ci_ds = compute_nodal_carbon_intensity(
    net, snapshots,
    dynamic_import_ci={},
    formulation='downstream',
    imports_accounting='territorial',
    checkpoint_path=None,
)
elapsed = time.time() - t0
print(f'Tracing : {elapsed:.1f} s')

ci_arr = ci_ds['carbon_intensity']
print(f'\nCI shape       : {ci_arr.shape}')
print(f'CI mean global : {float(ci_arr.mean()):.2f} gCO2/kWh')
print(f'CI median      : {float(np.nanmedian(ci_arr.values)):.2f}')
print(f'NaN            : {int(np.isnan(ci_arr.values).sum())}  ({100*np.isnan(ci_arr.values).mean():.1f}%)')

---
## 6. 🔬 Test de conservation du carbone

C'est LE test qui diagnostique le bug. Identité à vérifier :

$$\sum_g EF_g \cdot P_g(t) \;=\; \sum_l CI(bus_l, t) \cdot P_l(t)$$

C_produit total = C_consommé total. Si le ratio ≠ 1.0, le tracing perd (ou crée) du carbone.

In [ ]:
# Carbone produit (oracle)
C_prod_h = (net.generators_t.p_set * ef_vec).sum(axis=1)  # MW × g/kWh = g/h_equiv
C_prod_total = C_prod_h.sum()

# Carbone consommé via tracing
load_p = net.loads_t.p_set
C_load_h = pd.Series(0.0, index=snapshots)
for load_id in load_p.columns:
    bus = net.loads.loc[load_id, 'bus']
    if bus in ci_arr.coords['bus'].values:
        ci_at_load = ci_arr.sel(bus=bus).values
        C_load_h += load_p[load_id].values * ci_at_load
C_load_total = C_load_h.sum()

ratio = C_load_total / max(C_prod_total, 1)
loss_pct = (1 - ratio) * 100

print('─' * 60)
print('  CONSERVATION DU CARBONE — TEST PRINCIPAL')
print('─' * 60)
print(f'  C produit (Σ EF × P_gen)     : {C_prod_total/1e6:.2f} MgCO2')
print(f'  C consommé (Σ CI × P_load)   : {C_load_total/1e6:.2f} MgCO2')
print(f'  Ratio C_load / C_prod        : {ratio:.3f}  (cible = 1.000)')
print(f'  Carbone perdu                : {loss_pct:+.1f}%')
print()
if abs(loss_pct) < 2:
    print('  ✅ PASS : conservation ±2%')
else:
    print(f'  ❌ FAIL : {loss_pct:+.1f}% vs seuil ±2%. Localiser la fuite → cellules suivantes.')

In [ ]:
# CI nationale load-weighted (notre modèle) vs RTE
P_load_h = load_p.sum(axis=1)
ci_national_model = C_load_h / P_load_h.replace(0, np.nan)

ci_rte_window = ci_rte_series.reindex(snapshots).dropna()

print('── CI NATIONALE (load-weighted) ──')
print(f'  Modèle (après tracing) : mean={ci_national_model.mean():.2f}, median={ci_national_model.median():.2f}')
print(f'  RTE référence          : mean={ci_rte_window.mean():.2f}, median={ci_rte_window.median():.2f}')
print(f'  Biais                  : {ci_national_model.mean() - ci_rte_window.mean():+.2f} gCO2/kWh')
print(f'  MAPE horaire           : {100*(ci_national_model - ci_rte_window).abs().mean() / ci_rte_window.mean():.1f}%')

---
## 7. 🔍 Localiser la fuite carbone

Plusieurs hypothèses testables :
- **H1** : les imports (carriers `import_XX`) ne reçoivent pas de CI → le carbone qu'ils apportent est perdu
- **H2** : les bus sans load perdent le carbone (énergie injectée mais pas "consommée")
- **H3** : la résolution `A·x = P` donne des fractions x qui ne somment pas à 1

In [ ]:
# H3 — vérifier que Σ_k X[bus, k] ≈ 1 sur un snapshot
# Reprendre le code de trace_carbon_snapshot pour récupérer X_matrix
import scipy.sparse

t0 = snapshots[50]
line_flows = pd.Series(0.0, index=net.lines.index)
if hasattr(net, 'lines_t') and 'p0' in net.lines_t.__dict__ and t0 in net.lines_t.p0.index:
    line_flows = net.lines_t.p0.loc[t0]

nodal_inj = pd.Series(0.0, index=net.buses.index)
for gen_id, gen in net.generators.iterrows():
    bus = gen['bus']
    p = net.generators_t.p_set.loc[t0, gen_id] if gen_id in net.generators_t.p_set.columns else 0
    nodal_inj[bus] += max(p, 0)

A, bus_list = build_participation_matrix(net, t0, line_flows, nodal_inj)
print(f'Matrice A : {A.shape}, nnz={A.nnz}')

# Résoudre pour chaque carrier
bus_idx = {b: i for i, b in enumerate(bus_list)}
carriers = sorted(set(normalize_technology(c) for c in net.generators['carrier'].unique()))
P_gen_matrix = np.zeros((len(bus_list), len(carriers)))
for gen_id, gen in net.generators.iterrows():
    bus = gen['bus']
    if bus not in bus_idx:
        continue
    p = max(net.generators_t.p_set.loc[t0, gen_id] if gen_id in net.generators_t.p_set.columns else 0, 0)
    c = normalize_technology(str(gen.get('carrier', 'other')))
    if c in carriers:
        P_gen_matrix[bus_idx[bus], carriers.index(c)] += p

X_matrix = np.zeros_like(P_gen_matrix)
for i, c in enumerate(carriers):
    if P_gen_matrix[:, i].sum() > 0.01:
        X_matrix[:, i] = np.maximum(scipy.sparse.linalg.spsolve(A, P_gen_matrix[:, i]), 0)

# La somme de x sur tous les carriers devrait ≈ 1 pour chaque bus actif
X_sum = X_matrix.sum(axis=1)
active = X_sum > 0.01
print(f'\nBus actifs : {active.sum()} / {len(bus_list)}')
print(f'Σ_k X[bus, k] — mean : {X_sum[active].mean():.3f}  (cible = 1.0)')
print(f'Σ_k X[bus, k] — std  : {X_sum[active].std():.3f}')
print(f'Σ_k X[bus, k] — p50  : {np.percentile(X_sum[active], 50):.3f}')
print(f'Σ_k X[bus, k] — min  : {X_sum[active].min():.3f}')
print(f'Σ_k X[bus, k] — max  : {X_sum[active].max():.3f}')

if abs(X_sum[active].mean() - 1.0) < 0.05:
    print('\n  ✅ H3 OK — les fractions somment bien à 1')
else:
    print(f'\n  ❌ H3 FAIL — les fractions somment à {X_sum[active].mean():.3f} en moyenne')

In [ ]:
# Décomposition par carrier : où se retrouve chaque type de production dans les loads ?
carrier_contribution = {}
for i, c in enumerate(carriers):
    prod_c = P_gen_matrix[:, i].sum()
    traced_c = X_matrix[:, i].sum()
    ratio_c = traced_c / max(prod_c, 0.01)
    carrier_contribution[c] = (prod_c, traced_c, ratio_c)

print(f'── Conservation par carrier (snapshot {t0}) ──')
print(f'  {"carrier":25s} {"P_gen":>10s} {"P_traced":>10s} {"ratio":>8s}')
for c in sorted(carrier_contribution.keys(), key=lambda x: -carrier_contribution[x][0]):
    prod_c, traced_c, r = carrier_contribution[c]
    if prod_c > 1:
        print(f'  {c:25s} {prod_c:10.1f} {traced_c:10.1f} {r:8.3f}')

In [ ]:
# H2 — carbone qui arrive sur un bus SANS load est "perdu"
buses_with_load = set(net.loads['bus'])
active_buses = [bus_list[i] for i in range(len(bus_list)) if active[i]]

traced_total_active = sum(X_matrix[i, :].sum() for i in range(len(bus_list)) if active[i])
traced_on_load = sum(X_matrix[i, :].sum() for i in range(len(bus_list)) 
                     if active[i] and bus_list[i] in buses_with_load)

pct_on_load = 100 * traced_on_load / max(traced_total_active, 1)
print(f'Carbone tracé total (bus actifs) : {traced_total_active:.1f} MW-équivalent')
print(f'Carbone tracé sur bus avec load  : {traced_on_load:.1f} ({pct_on_load:.1f}%)')
print(f'Carbone tracé sur bus sans load  : {traced_total_active - traced_on_load:.1f} ({100-pct_on_load:.1f}%)')

if pct_on_load < 80:
    print(f'\n  ⚠ H2 : {100-pct_on_load:.1f}% du carbone tracé finit sur des bus sans load')
    print('     → le dispatch de load ne couvre pas tous les bus producteurs actifs')

---
## 8. 🧪 Tests A/B de correction

Une fois la fuite localisée, tester des fixes sur cette fenêtre de 168 h :
- **Fix 1** : redistribuer le carbone des bus sans load vers les loads voisins au prorata
- **Fix 2** : utiliser `formulation='upstream'` (au lieu de downstream)
- **Fix 3** : calculer la CI nationale directement (Σ CI × P / Σ P sur tout) plutôt que load-weighted

In [ ]:
# --- Fix 3 : CI nationale = C_prod / P_prod (ignore les loads) ---
# C'est la formule oracle. Elle contourne toute la problématique du tracing pour la CI nationale.
ci_national_direct = C_prod_h / prod_h.replace(0, np.nan)

print('── Comparaison CI nationale (méthode) ──')
print(f'  Oracle direct (C_prod/P_prod)        : {ci_national_direct.mean():.2f} gCO2/kWh')
print(f'  Load-weighted via tracing (modèle)   : {ci_national_model.mean():.2f} gCO2/kWh')
print(f'  RTE référence                        : {ci_rte_window.mean():.2f} gCO2/kWh')
print()
print(f'  Biais direct/RTE         : {ci_national_direct.mean() - ci_rte_window.mean():+.2f}')
print(f'  Biais tracing/RTE        : {ci_national_model.mean() - ci_rte_window.mean():+.2f}')

In [ ]:
# --- Fix 2 : formulation upstream ---
ci_ds_up = compute_nodal_carbon_intensity(
    net, snapshots, dynamic_import_ci={},
    formulation='upstream', imports_accounting='territorial', checkpoint_path=None,
)
ci_up = ci_ds_up['carbon_intensity']

C_load_up = pd.Series(0.0, index=snapshots)
for load_id in load_p.columns:
    bus = net.loads.loc[load_id, 'bus']
    if bus in ci_up.coords['bus'].values:
        C_load_up += load_p[load_id].values * ci_up.sel(bus=bus).values
ci_nat_up = C_load_up / P_load_h.replace(0, np.nan)

print('── Comparaison formulations ──')
print(f'  downstream : CI nat load-w = {ci_national_model.mean():.2f}, biais = {ci_national_model.mean()-ci_rte_window.mean():+.2f}')
print(f'  upstream   : CI nat load-w = {ci_nat_up.mean():.2f}, biais = {ci_nat_up.mean()-ci_rte_window.mean():+.2f}')
print(f'  RTE        : {ci_rte_window.mean():.2f}')

---
## 9. Visualisation — CI horaire modèle vs RTE

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.plot(snapshots, ci_rte_window.reindex(snapshots), label='RTE ref', color='black', lw=2)
ax.plot(snapshots, ci_national_direct, label='Oracle direct (Σ EF·P / Σ P)', color='green', ls='--')
ax.plot(snapshots, ci_national_model, label='Tracing downstream (load-weighted)', color='blue')
ax.plot(snapshots, ci_nat_up, label='Tracing upstream (load-weighted)', color='orange')
ax.set_ylabel('CI gCO2/kWh')
ax.set_title('CI nationale horaire — modèles vs RTE')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(snapshots, (ci_national_model - ci_rte_window.reindex(snapshots)), label='tracing - RTE', color='blue')
ax.plot(snapshots, (ci_national_direct - ci_rte_window.reindex(snapshots)), label='direct - RTE', color='green', ls='--')
ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('Écart gCO2/kWh')
ax.set_title('Écart horaire vs RTE')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 10. ✍️ Conclusions & prochaines actions

Remplir après exécution :

**Résultat Oracle direct** : ___ gCO2/kWh (attendu ~55)

**Résultat Tracing downstream load-weighted** : ___ gCO2/kWh

**Conservation carbone** : ratio ___ (attendu 1.00 ± 2%)

**Σ_k X[bus, k]** : ___ (attendu 1.00)

**% carbone qui finit sur bus avec load** : ___ (cible > 95%)

### Diagnostic

- [ ] Dispatch OK ?
- [ ] Oracle direct OK ?
- [ ] Tracing conserve ?
- [ ] X somme bien à 1 ?
- [ ] Loads couvrent tous les bus producteurs ?

### Fix à appliquer dans `carbon_tracing.py`

_à remplir_

### Critères de validation avant régénération full year

1. Bilan carbone conservé ±2% sur 168 h
2. CI nationale load-weighted ∈ [50, 60] gCO2/kWh
3. `validate_dataset.py` PASS sur `ci_nodal_2022.nc` régénéré